In [1]:
import os, sys
from pathlib import Path

# When the notebook lives in implementation_3/, switch to the project root so
# all 'data/...' relative paths resolve correctly. Also add implementation_3/
# to sys.path so the local modules (config, tier, xgboost_model, ...) are importable.
_notebook_dir = Path().resolve()
_project_root = _notebook_dir.parent
os.chdir(_project_root)
sys.path.insert(0, str(_notebook_dir))
print(f'Working directory: {Path.cwd()}')
print(f'Module path includes: {_notebook_dir}')


Working directory: /Users/advaychandramouli/dev/projects/summer
Module path includes: /Users/advaychandramouli/dev/projects/summer/implementation_3


# MIRA: Multimodal Infrastructure Risk Analyzer
This project is a POC for ACM Research;
Lead: Advay Chandramouli


## Import Requisite Libraries

In [2]:
import pandas as pd
df = pd.read_csv("data/im3_open_source_data_center_atlas.csv")
df.shape
df.head()

,id,state,state_abb,state_id,county,county_id,operator,ref,name,sqft,lon,lat,type
0,2744301,New Jersey,NJ,34,Middlesex County,23,NaN,NaN,Verizon,105786.0,-74.496521,40.544256,building
1,7805491,Ohio,OH,39,Franklin County,49,NaN,NaN,Discover Financial Services New Albany,188209.0,-82.814358,40.100657,building
2,9474864,North Carolina,NC,37,Caldwell County,27,Google,NaN,Google Data Center,3407194.0,-81.546515,35.894738,campus
3,13924557,Iowa,IA,19,Polk County,153,Microsoft,NaN,Project Alluvion,10962475.0,-93.711719,41.515955,campus
4,14593270,North Carolina,NC,37,Catawba County,35,NaN,NaN,Apple - Maiden Data Center,5431080.0,-81.261809,35.588771,campus


## Exploratory Data Analysis (EDA)

In [3]:
unique_operators = sorted(
    df["operator"].fillna("").replace("", "(Not specified)").unique()
)
print(f"{len(unique_operators)} unique operators:")
pd.DataFrame(unique_operators, columns=["operator"])

130 unique operators:


,operator
0,(Not specified)
1,AT&T
2,Actapio
3,AiNET
4,Alabama Supercomputer Authority
...,...
125,Yahoo
126,Yosemite Community College District
127,bigbyte.cc
128,datasite


In [4]:
import plotly.express as px

# Records by Operator (top 20 for readability; long names work best on horizontal bars)
operator_counts = (
    df["operator"]
    .fillna("")
    .replace("", "(Not specified)")
    .value_counts()
    .reset_index()
)
operator_counts.columns = ["operator", "count"]
operator_counts["pct"] = (
    operator_counts["count"] / operator_counts["count"].sum() * 100
).round(1)
operator_top = operator_counts.head(20).sort_values("count")
operator_top["label"] = operator_top.apply(
    lambda row: f"{row['count']} ({row['pct']}%)", axis=1
)

fig_ops = px.bar(
    operator_top,
    x="count",
    y="operator",
    orientation="h",
    title="Records by Operator (Top 20)",
    labels={"count": "Number of Data Centers", "operator": "Operator"},
    text="label",
    color="count",
    color_continuous_scale="Blues",
)
fig_ops.update_layout(
    yaxis={"categoryorder": "total ascending"},
    showlegend=False,
    coloraxis_showscale=False,
    height=640,
    margin={"l": 20, "r": 40, "t": 60, "b": 40},
    plot_bgcolor="white",
)
fig_ops.update_traces(textposition="outside", cliponaxis=False)
fig_ops.show()

# Records by State (horizontal bar chart with counts and percentages)
state_counts = df["state"].value_counts().reset_index()
state_counts.columns = ["state", "count"]
state_counts["pct"] = (state_counts["count"] / state_counts["count"].sum() * 100).round(
    1
)
state_sorted = state_counts.sort_values("count")
state_sorted["label"] = state_sorted.apply(
    lambda row: f"{row['count']} ({row['pct']}%)", axis=1
)

fig_states = px.bar(
    state_sorted,
    x="count",
    y="state",
    orientation="h",
    title="Records by State",
    labels={"count": "Number of Data Centers", "state": "State"},
    text="label",
    color="count",
    color_continuous_scale="Teal",
)
fig_states.update_layout(
    yaxis={"categoryorder": "total ascending"},
    showlegend=False,
    coloraxis_showscale=False,
    height=1100,
    margin={"l": 20, "r": 40, "t": 60, "b": 40},
    plot_bgcolor="white",
)
fig_states.update_traces(textposition="outside", cliponaxis=False)
fig_states.show()

## Tabular Preprocessing

In [5]:
df = df[df["type"] == "building"]
df_buildings = df.drop(columns=["state_abb", "state_id", "county", "county_id", "ref"])
df_buildings["sqft"] = df_buildings["sqft"].astype(int)

In [6]:
from config import (
    MW_DENSITY_W_PER_SQFT, MW_DIVISOR, TIER_LABELS,
    SQFT_COLO_MIN, SQFT_HYPER_MIN, MW_COLO_MIN, MW_HYPER_MIN,
    WRI_SENTINEL, RANDOM_SEED,
    WRI_SCORE_COLUMNS_ALL, WRI_SCORE_COLUMNS, ELP_SCORE_COLUMNS,
    SOURCE_METADATA_COLUMNS, DROP_COLUMNS, METADATA_COLUMNS,
    TARGET_COLUMN, FEATURE_COLUMNS_BY_SET,
)
from tier import assign_impact_tier

df_buildings['est_mw'] = df_buildings['sqft'] * MW_DENSITY_W_PER_SQFT / MW_DIVISOR
df_buildings['impact_tier'] = df_buildings.apply(
    lambda r: assign_impact_tier(r['sqft'], r['est_mw']), axis=1
)
df_buildings['impact_tier_label'] = df_buildings['impact_tier'].map(TIER_LABELS)


In [7]:
import plotly.express as px

tier_order = [TIER_LABELS[k] for k in sorted(TIER_LABELS)]

print("Impact tier distribution:")
print(df_buildings["impact_tier_label"].value_counts().reindex(tier_order).to_string())
print(f"\nTotal buildings: {len(df_buildings)}")

fig = px.histogram(
    df_buildings,
    x="est_mw",
    color="impact_tier_label",
    nbins=60,
    title="Estimated Power (MW) by Impact Tier",
    labels={"est_mw": "Estimated MW (150 W/sqft)", "impact_tier_label": "Tier"},
    category_orders={"impact_tier_label": tier_order},
    color_discrete_sequence=["#4C78A8", "#F58518", "#E45756"],
)
fig.update_layout(plot_bgcolor="white", height=400, margin={"t": 50, "b": 40})
fig.show()

Impact tier distribution:
impact_tier_label
Edge/Enterprise    181
Colocation         592
Hyperscale         267

Total buildings: 1040


In [8]:
print(f"Shape: {df_buildings.shape}")
print(f"Columns: {list(df_buildings.columns)}")
df_buildings.head()

Shape: (1040, 11)
Columns: ['id', 'state', 'operator', 'name', 'sqft', 'lon', 'lat', 'type', 'est_mw', 'impact_tier', 'impact_tier_label']


,id,state,operator,name,sqft,lon,lat,type,est_mw,impact_tier,impact_tier_label
0,2744301,New Jersey,NaN,Verizon,105786,-74.496521,40.544256,building,15.86790,1,Colocation
1,7805491,Ohio,NaN,Discover Financial Services New Albany,188209,-82.814358,40.100657,building,28.23135,1,Colocation
5,14930068,New Mexico,Sandia National Laboratories,HPC (880),158463,-106.542822,35.049942,building,23.76945,1,Colocation
7,15884451,New Jersey,Barclays,Barclays Datacenter,94373,-74.285481,40.643753,building,14.15595,1,Colocation
8,16282459,Maryland,AiNET,CyberNAP Glen Burnie,89879,-76.606431,39.140242,building,13.48185,1,Colocation


## WRI Aqueduct Input Format

In [9]:
example_coordinates = pd.read_csv("data/example_coordinates.csv")
print("Example coordinates format:")
example_coordinates

wri_aqueduct_df = df_buildings[["id", "name", "lat", "lon"]].copy()
wri_aqueduct_df.columns = example_coordinates.columns

print(f"Columns: {list(wri_aqueduct_df.columns)}")
print(f"Shape: {wri_aqueduct_df.shape}")
wri_aqueduct_df.head()

Example coordinates format:
Columns: ['id', 'location name', 'latitude (decimal degrees)', 'longitude (decimal degrees)']
Shape: (1040, 4)


,id,location name,latitude (decimal degrees),longitude (decimal degrees)
0,2744301,Verizon,40.544256,-74.496521
1,7805491,Discover Financial Services New Albany,40.100657,-82.814358
5,14930068,HPC (880),35.049942,-106.542822
7,15884451,Barclays Datacenter,40.643753,-74.285481
8,16282459,CyberNAP Glen Burnie,39.140242,-76.606431


In [10]:
from pathlib import Path

batch_dir = Path("data/wri_input_batches")
batch_dir.mkdir(parents=True, exist_ok=True)

batch_size = 500
batch_files = []

for batch_num, start in enumerate(range(0, len(wri_aqueduct_df), batch_size), start=1):
    batch_df = wri_aqueduct_df.iloc[start : start + batch_size].copy()
    batch_path = batch_dir / f"batch_{batch_num:03d}.csv"
    batch_df.to_csv(batch_path, index=False)
    batch_files.append((batch_path.name, len(batch_df)))

print(f"Created {len(batch_files)} batch(es) in {batch_dir}/")
for name, n_rows in batch_files:
    print(f"  {name}: {n_rows} rows")

Created 3 batch(es) in data/wri_input_batches/
  batch_001.csv: 500 rows
  batch_002.csv: 500 rows
  batch_003.csv: 40 rows


In [11]:
wri_top5_data = pd.read_csv("data/wri_top5_data.csv")

with open("data/columns.txt", "w") as f:
    f.write("\n".join(wri_top5_data.columns))

print(f"Columns ({len(wri_top5_data.columns)}):")
print(list(wri_top5_data.columns))

print("\nColumn types:")
wri_top5_data.dtypes

Columns (271):
['the_geom', 'points_id', 'location_name', 'input_address', 'match_address', 'latitude', 'longitude', 'major_basin_name', 'minor_basin_name', 'aquifer_name', 'string_id', 'aq30_id', 'pfaf_id', 'gid_1', 'aqid', 'gid_0', 'name_0', 'name_1', 'area_km2', 'bws_raw', 'bws_score', 'bws_cat', 'bws_label', 'bwd_raw', 'bwd_score', 'bwd_cat', 'bwd_label', 'iav_raw', 'iav_score', 'iav_cat', 'iav_label', 'sev_raw', 'sev_score', 'sev_cat', 'sev_label', 'gtd_raw', 'gtd_score', 'gtd_cat', 'gtd_label', 'rfr_raw', 'rfr_score', 'rfr_cat', 'rfr_label', 'cfr_raw', 'cfr_score', 'cfr_cat', 'cfr_label', 'drr_raw', 'drr_score', 'drr_cat', 'drr_label', 'ucw_raw', 'ucw_score', 'ucw_cat', 'ucw_label', 'cep_raw', 'cep_score', 'cep_cat', 'cep_label', 'udw_raw', 'udw_score', 'udw_cat', 'udw_label', 'usa_raw', 'usa_score', 'usa_cat', 'usa_label', 'rri_raw', 'rri_score', 'rri_cat', 'rri_label', 'w_awr_def_qan_raw', 'w_awr_def_qan_score', 'w_awr_def_qan_cat', 'w_awr_def_qan_label', 'w_awr_def_qan_weight_

the_geom                             str
points_id                          int64
location_name                        str
input_address                        str
match_address                        str
                                  ...   
w_awr_tex_tot_raw                float64
w_awr_tex_tot_score              float64
w_awr_tex_tot_cat                  int64
w_awr_tex_tot_label                  str
w_awr_tex_tot_weight_fraction    float64
Length: 271, dtype: object

## Combine WRI Source Batches

In [12]:
from pathlib import Path

source_dir = Path("data/wri_source_data")
batch_paths = sorted(source_dir.glob("batch_*.csv"))
if not batch_paths:
    raise FileNotFoundError(
        f"No WRI source batches found in {source_dir.resolve()}. "
        "Add batch_*.csv files before running the comparison."
    )

batch_dfs = [pd.read_csv(path) for path in batch_paths]
reference_columns = list(batch_dfs[0].columns)
assert all(list(batch.columns) == reference_columns for batch in batch_dfs), (
    "WRI batch schemas or column order differ"
)

# Preserve the complete Aqueduct response. Feature-specific views are derived later;
# this object must retain all source fields, including the ELP aggregate columns.
wri_source_full_df = pd.concat(batch_dfs, ignore_index=True)

print(f"Found {len(batch_paths)} batch file(s) in {source_dir}/")
for path, batch in zip(batch_paths, batch_dfs):
    print(f"  {path.name}: {len(batch)} rows, {len(batch.columns)} columns")
print(f"\nPreserved full response: {wri_source_full_df.shape}")
print(f"First columns: {reference_columns[:10]} ...")

Found 3 batch file(s) in data/wri_source_data/
  batch_001.csv: 500 rows, 271 columns
  batch_002.csv: 500 rows, 271 columns
  batch_003.csv: 40 rows, 271 columns

Preserved full response: (1040, 271)
First columns: ['the_geom', 'points_id', 'location_name', 'input_address', 'match_address', 'latitude', 'longitude', 'major_basin_name', 'minor_basin_name', 'aquifer_name'] ...


In [13]:
# WRI_SCORE_COLUMNS_ALL, ELP_SCORE_COLUMNS, and SOURCE_METADATA_COLUMNS
# are imported from config.py. Validate that every required column is
# present in the full Aqueduct response loaded above.
required_source_columns = SOURCE_METADATA_COLUMNS + WRI_SCORE_COLUMNS_ALL + ELP_SCORE_COLUMNS
missing_source_columns = sorted(
    set(required_source_columns) - set(wri_source_full_df.columns)
)
assert not missing_source_columns, f'Missing required WRI/ELP columns: {missing_source_columns}'

expected_ids = set(wri_aqueduct_df['id'])
actual_ids   = set(wri_source_full_df['points_id'])
duplicate_ids = wri_source_full_df.loc[
    wri_source_full_df['points_id'].duplicated(keep=False), 'points_id'
].unique()

print('Source alignment checks')
print(f'  input rows:              {len(wri_aqueduct_df)}')
print(f'  response rows:           {len(wri_source_full_df)}')
print(f'  missing IDs:             {len(expected_ids - actual_ids)}')
print(f'  extra IDs:               {len(actual_ids - expected_ids)}')
print(f'  duplicated response IDs: {len(duplicate_ids)}')

wri_source_df = wri_source_full_df[required_source_columns].copy()
from pathlib import Path
output_path = Path('data/wri_source_model_columns.csv')
wri_source_df.to_csv(output_path, index=False)
print(f'Saved modeling-column view to {output_path} ({wri_source_df.shape})')


Source alignment checks
  input rows:              1040
  response rows:           1040
  missing IDs:             0
  extra IDs:               0
  duplicated response IDs: 1
Saved modeling-column view to data/wri_source_model_columns.csv ((1040, 21))


In [14]:
import numpy as np

WRI_SENTINEL = -9999.0
ALL_CANDIDATE_FEATURES = WRI_SCORE_COLUMNS_ALL + ELP_SCORE_COLUMNS

quality_records = []
for col in ALL_CANDIDATE_FEATURES:
    raw = wri_source_full_df[col]
    numeric = pd.to_numeric(raw, errors="coerce")
    valid = numeric.replace([WRI_SENTINEL, np.inf, -np.inf], np.nan).dropna()
    quality_records.append({
        "feature_set": "ELP aggregates" if col in ELP_SCORE_COLUMNS else "WRI indicators",
        "column": col,
        "source_dtype": str(raw.dtype),
        "non_numeric_count": int((raw.notna() & numeric.isna()).sum()),
        "missing_count": int(numeric.isna().sum()),
        "missing_pct": 100 * numeric.isna().mean(),
        "sentinel_count": int((numeric == WRI_SENTINEL).sum()),
        "sentinel_pct": 100 * (numeric == WRI_SENTINEL).mean(),
        "n_valid": int(valid.size),
        "variance": float(valid.var()) if valid.size else np.nan,
        "min": float(valid.min()) if valid.size else np.nan,
        "max": float(valid.max()) if valid.size else np.nan,
    })

feature_quality = pd.DataFrame(quality_records).set_index("column")
assert (feature_quality["non_numeric_count"] == 0).all(), "Non-numeric feature values found"
assert (feature_quality["n_valid"] > 0).all(), "A required feature has no valid values"

print("WRI and ELP feature diagnostics")
display(feature_quality.round(4))
print("\nELP diagnostics:")
display(feature_quality.loc[ELP_SCORE_COLUMNS].round(4))


WRI and ELP feature diagnostics


,feature_set,source_dtype,non_numeric_count,missing_count,missing_pct,sentinel_count,sentinel_pct,n_valid,variance,min,max
column,,,,,,,,,,,
bws_score,WRI indicators,float64,0,0,0.0,0,0.0,1040,3.1113,0.0000,5.0000
bwd_score,WRI indicators,float64,0,0,0.0,0,0.0,1040,2.0131,0.0040,5.0000
iav_score,WRI indicators,float64,0,0,0.0,0,0.0,1040,0.7140,0.4399,4.7401
sev_score,WRI indicators,float64,0,0,0.0,0,0.0,1040,0.3867,0.1413,2.9154
gtd_score,WRI indicators,float64,0,0,0.0,780,75.0,260,0.5756,1.0644,3.2979
rfr_score,WRI indicators,float64,0,0,0.0,0,0.0,1040,0.7555,0.0001,3.4068
cfr_score,WRI indicators,float64,0,0,0.0,0,0.0,1040,0.2460,0.0000,2.6971
drr_score,WRI indicators,float64,0,0,0.0,0,0.0,1040,0.1380,1.0109,2.9249
ucw_score,WRI indicators,float64,0,0,0.0,0,0.0,1040,0.0000,0.8807,0.8807



ELP diagnostics:


,feature_set,source_dtype,non_numeric_count,missing_count,missing_pct,sentinel_count,sentinel_pct,n_valid,variance,min,max
column,,,,,,,,,,,
w_awr_elp_qan_score,ELP aggregates,float64,0,0,0.0,0,0.0,1040,1.3122,0.6034,4.2268
w_awr_elp_qal_score,ELP aggregates,float64,0,0,0.0,0,0.0,1040,0.9001,0.6436,3.7564
w_awr_elp_rrr_score,ELP aggregates,float64,0,0,0.0,0,0.0,1040,0.0234,0.9778,1.4255
w_awr_elp_tot_score,ELP aggregates,float64,0,0,0.0,0,0.0,1040,0.9111,0.4665,4.0868


### Feature definitions and pruning

The **WRI indicators** representation retains the nine previously selected risk indicators. `gtd_score` is excluded for its high sentinel rate; `ucw_score` and `rri_score` are constant; and `usa_score` has negligible variance.

The **ELP aggregates** representation contains the four electric-power-weighted Aqueduct scores requested for the comparison: quantity, quality, regulatory/reputational risk, and total risk. The total is derived from related dimensions, so it is expected to be correlated with the three components. Native and permutation importances are therefore interpreted as importance within a redundant feature group, not as independent causal effects.


In [15]:
# DROP_COLUMNS, WRI_SCORE_COLUMNS, and FEATURE_COLUMNS_BY_SET are defined in config.py.
# Assert quality constraints still hold for this dataset.
assert len(WRI_SCORE_COLUMNS) == 9
assert (feature_quality.loc[WRI_SCORE_COLUMNS, 'variance'] > 0).all()
assert (feature_quality.loc[ELP_SCORE_COLUMNS, 'variance'] > 0).all()

print(f'Dropped from WRI representation: {DROP_COLUMNS}')
for feature_set, columns in FEATURE_COLUMNS_BY_SET.items():
    print(f'{feature_set} ({len(columns)}): {columns}')


Dropped from WRI representation: ['gtd_score', 'ucw_score', 'rri_score', 'usa_score']
WRI indicators (9): ['bws_score', 'bwd_score', 'iav_score', 'sev_score', 'rfr_score', 'cfr_score', 'drr_score', 'cep_score', 'udw_score']
ELP aggregates (4): ['w_awr_elp_qan_score', 'w_awr_elp_qal_score', 'w_awr_elp_rrr_score', 'w_awr_elp_tot_score']


## Build ID-aligned WRI and ELP feature sets

Both representations use exactly the same observations and target labels. The five confirmed metadata fields—`id`, `name`, `lon`, `lat`, and `impact_tier`—are retained for traceability but never passed to either model.

In [16]:
# METADATA_COLUMNS and TARGET_COLUMN imported from config.py

source_for_join = wri_source_full_df.rename(columns={
    "points_id": "id",
    "location_name": "name",
    "longitude": "lon",
    "latitude": "lat",
})
source_for_join = source_for_join[
    ["id", "name", "lon", "lat"] + WRI_SCORE_COLUMNS + ELP_SCORE_COLUMNS
].copy()

building_meta = (
    df_buildings[METADATA_COLUMNS]
    .drop_duplicates(subset="id", keep="first")
    .copy()
)
source_duplicates = source_for_join[source_for_join["id"].duplicated(keep=False)]
source_for_join = source_for_join.drop_duplicates(subset="id", keep="first")

# Building metadata is authoritative; compare source metadata before selecting features.
metadata_check = building_meta.merge(
    source_for_join[["id", "name", "lon", "lat"]],
    on="id",
    how="inner",
    suffixes=("_building", "_wri"),
    validate="one_to_one",
)
name_match = (
    metadata_check["name_building"].fillna("").astype(str).str.strip()
    == metadata_check["name_wri"].fillna("").astype(str).str.strip()
)
coord_match = np.isclose(
    metadata_check[["lon_building", "lat_building"]],
    metadata_check[["lon_wri", "lat_wri"]].to_numpy(),
    atol=1e-6,
    equal_nan=True,
).all(axis=1)
metadata_mismatches = metadata_check.loc[~(name_match & coord_match)]

all_features = WRI_SCORE_COLUMNS + ELP_SCORE_COLUMNS
for col in all_features:
    source_for_join[col] = pd.to_numeric(source_for_join[col], errors="coerce")
source_for_join[all_features] = source_for_join[all_features].replace(
    [WRI_SENTINEL, np.inf, -np.inf], np.nan
)

aligned = building_meta.merge(
    source_for_join[["id"] + all_features],
    on="id",
    how="inner",
    validate="one_to_one",
).sort_values("id").reset_index(drop=True)

# Complete cases are selected jointly, never separately by feature representation.
shared_complete_mask = aligned[all_features].notna().all(axis=1)
aligned = aligned.loc[shared_complete_mask].reset_index(drop=True)
df_ml_wri = aligned[METADATA_COLUMNS + WRI_SCORE_COLUMNS].copy()
df_ml_elp = aligned[METADATA_COLUMNS + ELP_SCORE_COLUMNS].copy()

assert list(df_ml_elp.columns) == METADATA_COLUMNS + ELP_SCORE_COLUMNS
assert df_ml_wri["id"].is_unique and df_ml_elp["id"].is_unique
assert df_ml_wri["id"].equals(df_ml_elp["id"])
assert df_ml_wri[TARGET_COLUMN].equals(df_ml_elp[TARGET_COLUMN])

MODEL_DATAFRAMES = {
    "WRI indicators": df_ml_wri,
    "ELP aggregates": df_ml_elp,
}
# Compatibility alias for earlier WRI-only exploration.
df_ml = df_ml_wri

print(f"Deduplicated source rows: {len(source_duplicates)} rows across "
      f"{source_duplicates['id'].nunique()} ID(s)")
print(f"Metadata mismatches on shared IDs: {len(metadata_mismatches)}")
print(f"Rows excluded jointly for incomplete/non-finite features: {(~shared_complete_mask).sum()}")
for feature_set, frame in MODEL_DATAFRAMES.items():
    print(f"{feature_set}: {frame.shape}; feature shape "
          f"{frame[FEATURE_COLUMNS_BY_SET[feature_set]].shape}")
display(df_ml_elp.head())

Deduplicated source rows: 2 rows across 1 ID(s)
Metadata mismatches on shared IDs: 4
Rows excluded jointly for incomplete/non-finite features: 0
WRI indicators: (1039, 14); feature shape (1039, 9)
ELP aggregates: (1039, 9); feature shape (1039, 4)


,id,name,lon,lat,impact_tier,w_awr_elp_qan_score,w_awr_elp_qal_score,w_awr_elp_rrr_score,w_awr_elp_tot_score
0,2744301,Verizon,-74.496521,40.544256,1,2.945868,3.534384,1.050924,2.045674
1,7805491,Discover Financial Services New Albany,-82.814358,40.100657,1,2.526726,2.450120,1.077413,1.633669
2,14930068,HPC (880),-106.542822,35.049942,1,3.026504,1.443104,1.155910,1.968278
3,15884451,Barclays Datacenter,-74.285481,40.643753,1,2.827211,3.238904,0.977778,1.915753
4,16282459,CyberNAP Glen Burnie,-76.606431,39.140242,1,1.910742,3.580165,0.977778,1.249638


In [17]:
model_frames = {}
for feature_set, frame in MODEL_DATAFRAMES.items():
    feature_columns = FEATURE_COLUMNS_BY_SET[feature_set]
    model_frame = frame[[TARGET_COLUMN] + feature_columns].copy()
    assert list(model_frame.columns) == [TARGET_COLUMN] + feature_columns
    assert not set(METADATA_COLUMNS).intersection(feature_columns), (
        "Metadata leakage detected in predictors"
    )
    assert model_frame[feature_columns].notna().all().all()
    assert np.isfinite(model_frame[feature_columns].to_numpy()).all()
    assert not (model_frame[feature_columns] == WRI_SENTINEL).any().any()
    model_frames[feature_set] = model_frame

shared_ids = df_ml_wri["id"].copy()
shared_target = df_ml_wri[TARGET_COLUMN].copy()
assert shared_target.equals(df_ml_elp[TARGET_COLUMN])

print(f"Shared model rows: {len(shared_ids)}")
print("Predictor counts:", {
    name: len(columns) for name, columns in FEATURE_COLUMNS_BY_SET.items()
})
print("Metadata retained but excluded from predictors:", METADATA_COLUMNS)
print("\nShared target distribution:")
print(shared_target.map(TIER_LABELS).value_counts().sort_index().to_string())


Shared model rows: 1039
Predictor counts: {'WRI indicators': 9, 'ELP aggregates': 4}
Metadata retained but excluded from predictors: ['id', 'name', 'lon', 'lat', 'impact_tier']

Shared target distribution:
impact_tier
Colocation         592
Edge/Enterprise    181
Hyperscale         266


## Fair WRI-vs-ELP model comparison

The experiment uses one stratified 80/20 split of row IDs and one fixed five-fold stratified CV assignment for every feature-set/model/configuration pair. This holds observations, targets, random seeds, train/test membership, and validation folds constant. Hyperparameters are selected by mean CV macro-F1 on the shared training rows; the untouched shared test rows are used once for final comparison.


In [18]:
from sklearn.model_selection import StratifiedKFold, train_test_split

# RANDOM_SEED imported from config.py
train_ids, test_ids = train_test_split(
    shared_ids,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=shared_target,
)
train_id_set, test_id_set = set(train_ids), set(test_ids)
assert train_id_set.isdisjoint(test_id_set)
assert train_id_set | test_id_set == set(shared_ids)

train_positions = np.flatnonzero(shared_ids.isin(train_id_set).to_numpy())
test_positions = np.flatnonzero(shared_ids.isin(test_id_set).to_numpy())
y_train = shared_target.iloc[train_positions].reset_index(drop=True)
y_test = shared_target.iloc[test_positions].reset_index(drop=True)

cv_splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
shared_cv_splits = [
    (train_idx.copy(), valid_idx.copy())
    for train_idx, valid_idx in cv_splitter.split(np.zeros(len(y_train)), y_train)
]

split_data = {}
for feature_set, frame in MODEL_DATAFRAMES.items():
    columns = FEATURE_COLUMNS_BY_SET[feature_set]
    split_data[feature_set] = {
        "X_train": frame.iloc[train_positions][columns].reset_index(drop=True),
        "X_test": frame.iloc[test_positions][columns].reset_index(drop=True),
        "y_train": y_train,
        "y_test": y_test,
    }

print(f"Shared train rows: {len(train_positions)} | shared test rows: {len(test_positions)}")
print("Shared five-fold validation sizes:", [len(valid) for _, valid in shared_cv_splits])
print("\nClass distribution:")
for label, tier_name in sorted(TIER_LABELS.items()):
    n_train = int((y_train == label).sum())
    n_test = int((y_test == label).sum())
    print(f"  {tier_name:<20} train={n_train:>3} ({n_train/len(y_train):.1%}) "
          f"test={n_test:>3} ({n_test/len(y_test):.1%})")

Shared train rows: 831 | shared test rows: 208
Shared five-fold validation sizes: [167, 166, 166, 166, 166]

Class distribution:
  Edge/Enterprise      train=145 (17.4%) test= 36 (17.3%)
  Colocation           train=473 (56.9%) test=119 (57.2%)
  Hyperscale           train=213 (25.6%) test= 53 (25.5%)


In [19]:
assert set(y_train.unique()).issubset({0, 1, 2})
assert set(y_test.unique()).issubset({0, 1, 2})
assert df_ml_wri.iloc[train_positions]["id"].tolist() == df_ml_elp.iloc[train_positions]["id"].tolist()
assert df_ml_wri.iloc[test_positions]["id"].tolist() == df_ml_elp.iloc[test_positions]["id"].tolist()

quality_summary = []
for feature_set, data in split_data.items():
    X_train = data["X_train"]
    X_test = data["X_test"]
    assert X_train.shape == (len(train_positions), len(FEATURE_COLUMNS_BY_SET[feature_set]))
    assert X_test.shape == (len(test_positions), len(FEATURE_COLUMNS_BY_SET[feature_set]))
    assert X_train.index.equals(y_train.index) and X_test.index.equals(y_test.index)
    for split_name, X_part in [("train", X_train), ("test", X_test)]:
        quality_summary.append({
            "feature_set": feature_set,
            "split": split_name,
            "rows": len(X_part),
            "features": X_part.shape[1],
            "nan": int(X_part.isna().sum().sum()),
            "sentinel": int((X_part == WRI_SENTINEL).sum().sum()),
            "infinite": int(np.isinf(X_part.to_numpy()).sum()),
        })

split_quality = pd.DataFrame(quality_summary)
assert split_quality[["nan", "sentinel", "infinite"]].to_numpy().sum() == 0
print("Fair-comparison invariants passed: identical IDs, targets, split, folds, seed, and clean predictors.")
display(split_quality)

Fair-comparison invariants passed: identical IDs, targets, split, folds, seed, and clean predictors.


,feature_set,split,rows,features,nan,sentinel,infinite
0,WRI indicators,train,831,9,0,0,0
1,WRI indicators,test,208,9,0,0,0
2,ELP aggregates,train,831,4,0,0,0
3,ELP aggregates,test,208,4,0,0,0


In [20]:
# Shared artefact dicts — populated by train_xgboost and train_catboost
trained_models = {}
selected_configs = {}
cv_score_distributions = {}


In [21]:
from xgboost_model import train_xgboost, XGB_CONFIGS

xgb_trained, xgb_selected, xgb_cv_dists, xgb_search_results = train_xgboost(
    split_data, shared_cv_splits, FEATURE_COLUMNS_BY_SET, RANDOM_SEED
)
trained_models.update(xgb_trained)
selected_configs.update(xgb_selected)
cv_score_distributions.update(xgb_cv_dists)

display(xgb_search_results.sort_values(
    ['feature_set', 'cv_macro_f1_mean'], ascending=[True, False]
))
print('Selected XGBoost configs:', {
    fs: xgb_selected[(fs, 'XGBoost')] for fs in FEATURE_COLUMNS_BY_SET
})


,feature_set,config,cv_macro_f1_mean,cv_macro_f1_std
7,ELP aggregates,regularized,0.473770,0.035116
6,ELP aggregates,deep_fast,0.462688,0.040136
5,ELP aggregates,balanced,0.455373,0.027213
4,ELP aggregates,shallow_slow,0.449563,0.035293
2,WRI indicators,deep_fast,0.443959,0.033108
1,WRI indicators,balanced,0.443489,0.034363
3,WRI indicators,regularized,0.441631,0.035315
0,WRI indicators,shallow_slow,0.439679,0.043062


Selected XGBoost configs: {'WRI indicators': 'deep_fast', 'ELP aggregates': 'regularized'}


In [22]:
from catboost_model import train_catboost, CB_CONFIGS

cb_trained, cb_selected, cb_cv_dists, cb_search_results = train_catboost(
    split_data, shared_cv_splits, FEATURE_COLUMNS_BY_SET, RANDOM_SEED
)
trained_models.update(cb_trained)
selected_configs.update(cb_selected)
cv_score_distributions.update(cb_cv_dists)

display(cb_search_results.sort_values(
    ['feature_set', 'cv_macro_f1_mean'], ascending=[True, False]
))
print('Selected CatBoost configs:', {
    fs: cb_selected[(fs, 'CatBoost')] for fs in FEATURE_COLUMNS_BY_SET
})


,feature_set,config,cv_macro_f1_mean,cv_macro_f1_std
5,ELP aggregates,balanced,0.462941,0.031633
6,ELP aggregates,deep_fast,0.460186,0.038582
7,ELP aggregates,regularized,0.454930,0.022953
4,ELP aggregates,shallow_slow,0.452474,0.025638
1,WRI indicators,balanced,0.441030,0.041101
2,WRI indicators,deep_fast,0.436624,0.031228
0,WRI indicators,shallow_slow,0.431442,0.033460
3,WRI indicators,regularized,0.430468,0.037257


Selected CatBoost configs: {'WRI indicators': 'balanced', 'ELP aggregates': 'balanced'}


In [23]:
from evaluation import compute_comparison_results
import plotly.figure_factory as ff

comparison_results, test_predictions, confusion_matrices = compute_comparison_results(
    split_data, trained_models, selected_configs, cv_score_distributions,
    FEATURE_COLUMNS_BY_SET, y_train, y_test, shared_cv_splits,
)

tier_names = [TIER_LABELS[k] for k in sorted(TIER_LABELS)]
print('Primary metric: macro-F1. Test metrics use the same held-out IDs for every row.')
display(comparison_results.round(3))

for (feature_set, model_name), matrix in confusion_matrices.items():
    fig = ff.create_annotated_heatmap(
        matrix, x=tier_names, y=tier_names, colorscale='Blues', showscale=True,
    )
    fig.update_layout(
        title=f'{feature_set} · {model_name} — shared test confusion matrix',
        xaxis_title='Predicted', yaxis_title='Actual',
        yaxis={'autorange': 'reversed'}, height=420, margin={'t': 70},
    )
    fig.show()


Primary metric: macro-F1. Test metrics use the same held-out IDs for every row.


,feature_set,model,selected_config,cv_macro_f1_mean,cv_macro_f1_std,test_macro_f1,test_weighted_f1,test_accuracy,f1_edge_enterprise,f1_colocation,f1_hyperscale
0,ELP aggregates,XGBoost,regularized,0.474,0.035,0.511,0.556,0.548,0.415,0.630,0.487
1,ELP aggregates,CatBoost,balanced,0.463,0.032,0.481,0.547,0.543,0.306,0.643,0.496
2,WRI indicators,CatBoost,balanced,0.441,0.041,0.473,0.526,0.519,0.333,0.604,0.483
3,WRI indicators,XGBoost,deep_fast,0.444,0.033,0.458,0.525,0.524,0.261,0.617,0.496
4,Shared rows,Majority baseline,predict Colocation,0.242,0.001,0.243,0.416,0.572,0.000,0.728,0.000


In [24]:
from evaluation import compute_feature_importance
import plotly.express as px

feature_importance_results = compute_feature_importance(
    split_data, trained_models, FEATURE_COLUMNS_BY_SET,
    random_seed=RANDOM_SEED, n_repeats=30,
)

display(feature_importance_results.sort_values(
    ['feature_set', 'model', 'method', 'rank_within_method']
).round(4))

for (feature_set, model_name, method), ranking in feature_importance_results.groupby(
    ['feature_set', 'model', 'method'], sort=False
):
    ranking = ranking.sort_values('importance', ascending=True)
    fig = px.bar(
        ranking, x='importance', y='feature', orientation='h',
        error_x='importance_std' if method.startswith('Permutation') else None,
        title=f'{feature_set} · {model_name} — {method} importance',
        labels={'importance': 'Importance', 'feature': 'Feature'},
    )
    fig.update_layout(plot_bgcolor='white', height=430, margin={'l': 170, 't': 70})
    fig.show()

elp_correlations = df_ml_elp[ELP_SCORE_COLUMNS].corr()
print('ELP aggregate correlations (shared rows):')
display(elp_correlations.round(3))
print(
    'Interpretation warning: w_awr_elp_tot_score aggregates related quantity, quality, '
    'and regulatory/reputational dimensions. Correlation and interchangeable split credit '
    'can dilute or redistribute both native and permutation importance; rankings are not '
    'independent-effect estimates.'
)


,feature_set,model,method,feature,importance,importance_std,rank_within_method
48,ELP aggregates,CatBoost,Native,w_awr_elp_rrr_score,27.9953,NaN,1
46,ELP aggregates,CatBoost,Native,w_awr_elp_qal_score,27.6073,NaN,2
50,ELP aggregates,CatBoost,Native,w_awr_elp_tot_score,23.1674,NaN,3
44,ELP aggregates,CatBoost,Native,w_awr_elp_qan_score,21.2300,NaN,4
47,ELP aggregates,CatBoost,Permutation (test macro-F1),w_awr_elp_qal_score,0.0916,0.0257,1
49,ELP aggregates,CatBoost,Permutation (test macro-F1),w_awr_elp_rrr_score,0.0861,0.0188,2
45,ELP aggregates,CatBoost,Permutation (test macro-F1),w_awr_elp_qan_score,0.0386,0.0198,3
51,ELP aggregates,CatBoost,Permutation (test macro-F1),w_awr_elp_tot_score,0.0280,0.0225,4
40,ELP aggregates,XGBoost,Native,w_awr_elp_rrr_score,0.2815,NaN,1
38,ELP aggregates,XGBoost,Native,w_awr_elp_qal_score,0.2526,NaN,2


ELP aggregate correlations (shared rows):


,w_awr_elp_qan_score,w_awr_elp_qal_score,w_awr_elp_rrr_score,w_awr_elp_tot_score
w_awr_elp_qan_score,1.000,-0.311,0.068,0.955
w_awr_elp_qal_score,-0.311,1.000,-0.477,-0.247
w_awr_elp_rrr_score,0.068,-0.477,1.000,0.109
w_awr_elp_tot_score,0.955,-0.247,0.109,1.000


Interpretation warning: w_awr_elp_tot_score aggregates related quantity, quality, and regulatory/reputational dimensions. Correlation and interchangeable split credit can dilute or redistribute both native and permutation importance; rankings are not independent-effect estimates.
